In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.\
            builder.\
                appName('Window Fucntion').\
                    getOrCreate()

In [0]:
emp_schema = "emp_id INT, dept_id INT, name STRING, age INT, gender STRING, salary DOUBLE, hire_date STRING"

emp_data = [
    (1, 101, "John Smith", 30, "Male", 85000.0, "2021-03-15"),
    (2, 101, "Alice Johnson", 35, "Female", 92000.0, "2020-07-01"),
    (3, 102, "Bob Williams", 28, None, 65000.0, "2022-01-20"),
    (4, 102, "Carol Brown", 40, "Female", 68000.0, "2019-11-05"),
    (5, 103, "David Jones", 32, "Male", 55000.0, "2021-09-10"),
    (6, 103, "Eva Garcia", 26, "Female", 58000.0, "2023-02-28"),
    (7, 101, "Frank Miller", 45, "Male", 78000.0, "2018-05-22"),
    (8, 104, "Grace Davis", 29, "Female", 72000.0, "2022-08-14"),
    (9, 104, "Henry Wilson", 33, "Other", 70000.0, "2020-12-01"),
    (10, 105, "Ivy Taylor", 50, "Female", 95000.0, "2017-06-30"),
    (11, 105, "Jack Anderson", 38, "Male", 89000.0, "2021-04-18"),
    (12, 101, "Karen Thomas", 25, "Female", 81000.0, "2023-03-07"),
    (13, 102, "Leo Moore", 37, None, 62000.0, "2019-10-25"),
    (14, 101, "Mia Martin", 27, "Female", 90000.0, "2022-05-11"),
    (15, 103, "Nick Lee", 42, "Male", 60000.0, "2018-02-09"),
    (16, 104, "Olivia Harris", 31, "Female", 74000.0, "2020-09-03"),
    (17, 105, "Paul Clark", 24, "Male", 87000.0, "2023-07-19"),
    (18, 101, "Quinn Lewis", 34, "Other", 83000.0, "2021-12-12"),
    (19, 102, "Rachel Walker", 36, "Female", 66000.0, "2019-08-08"),
    (20, 104, "Sam Hall", 23, None, 71000.0, "2022-11-27"),
]

## Creating the employee DataFrame
emp = spark.createDataFrame(
    data=emp_data,
    schema=emp_schema
)

## Department dataset
dept_schema = "dept_id INT, dept_name STRING, city STRING, country STRING, budget DOUBLE"

dept_data = [
    (101, "Engineering", "New York", "USA", 2500000.0),
    (102, "Sales", "London", "UK", 1200000.0),
    (103, "HR", "Toronto", "Canada", 800000.0),
    (104, "Marketing", "Berlin", "Germany", 1500000.0),
    (105, "Finance", "Mumbai", "India", 2000000.0),
]

dept = spark.createDataFrame(
    data=dept_data,
    schema=dept_schema
)

In [0]:
emp.show()
dept.show()

emp.printSchema()
dept.printSchema()

In [0]:
# ## nummber of partition
# emp.rdd.getNumPartittions()
# dept.rdd.getNumPartittions()

# ## repartition
# ## it is usually use to increase the number of partition, more costly due to the complete shuffle of data
# emp = emp.repartition(4)
# dept = dept.repartition(4)
# emp.rdd.getNumPartittions()

# ## coalesce
# ## it is usually used to decrease thte number of partitions, less costlier than repartition due to no shuffling
# emp = emp.coalesce(2)
# dept = dept.coalesce(2)

**Note**: above cell might not work in dataBricks cuz in serverless/free tier we don't have much options or freedom of our code to run as per our choice.

In [0]:
## spark_partition_id
from pyspark.sql.functions import spark_partition_id

emp_partition_id = emp.withColumn("partition_num", spark_partition_id())
emp_partition_id.show()

In [0]:
# we can also repartition the data based on column, so that particular column will be on same partition
# partitioning on department id

emp_part_dept = emp.repartition(3, 'dept_id').\
                        withColumn('partition_num', spark_partition_id())

emp_part_dept.show()
# emp_part_dept.rdd.getNumPartittions()

In [0]:
## joins in pyspark
# inner join
emp_inner = emp.join(dept, how='inner', on='dept_id')
emp_inner.show()

In [0]:
# left join
emp_left = emp.join(dept, how='left', on='dept_id')
emp_left.show()

In [0]:
# right join
emp_right = emp.join(dept, how='right', on='dept_id')
emp_right.show()

In [0]:
# outer join
emp_outer = emp.join(dept, how='outer', on='dept_id')
emp_outer.show()

In [0]:
# left semi join
'''
In PySpark, a Semi Join (Left Semi Join) returns only the rows from the left DataFrame that have a matching record in the right DataFrame.
The important thing is:
    It returns columns only from the LEFT DataFrame, not from the RIGHT DataFrame.
'''
emp_leftsemi = emp.join(dept, how='leftsemi', on='dept_id')
emp_leftsemi.show()

In [0]:
# left anti join
'''
In PySpark, a Left Anti Join returns only those records from the LEFT DataFrame that do NOT have a matching record in the RIGHT DataFrame.
In simple words:
    Anti Join = Find records in the left table that do not exist in the right table.
'''
emp_leftanti = emp.join(dept, how='leftanti', on='dept_id')
emp_leftanti.show()

In [0]:
# cross join
emp_cross = emp.crossJoin(dept)
emp_cross.show()

In [0]:
## join with cascading conditions
# join with depaetment id and with id 101 and 102
emp_cascade = emp.join(dept, how='left', 
                       on=(
                            (emp.dept_id == dept.dept_id) & 
                            (emp.dept_id.isin(101, 102))
                       ))

emp_cascade.show()

In [0]:
emp_part_default = emp.repartition(3).\
                        withColumn('partition_num', spark_partition_id())

emp_part_default.show()
emp_part_default.groupBy('partition_num').count().show()
